# Extraction des coordonnées MediaPipe
Ce notebook extrait les points clés du corps sur chaque image et les sauvegarde dans un CSV.
Ici une sélection est faite sur les points clés les plus utiles.

## Imports

In [ ]:
import mediapipe as mp
import cv2
import pandas as pd 
import numpy as np
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# CONFIGURATION
DATASET_DIR = 'dataset'
CLASSES     = ['ap_tchagui', 'yop_tchagui']
OUTPUT_CSV  = 'exports/pose_coordonnes.csv'
MODEL_PATH  = 'models/pose_landmarker.task' # Modèle Media Pipe

print(f'   MediaPipe version : {mp.__version__}')

   MediaPipe version : 0.10.35


## Chargement du modèle

In [ ]:
# Configuration de base du modèle
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)

# Paramètres de détection de pose
options      = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE, # Traitement image par image
    min_pose_detection_confidence=0.3, # Seuil minimal pour accepter une détection
    min_pose_presence_confidence=0.3, # Seuil minimal de présence du corps détecté       
    min_tracking_confidence=0.3, # Seuil minimal de suivi des landmarks
)

print(f'Modèle chargé : {MODEL_PATH}')

Modèle chargé : models/pose_landmarker.task


## Noms des coordonnées

In [3]:
# Liste officielle des 33 landmarks du modèle Pose Landmarker de MediaPipe :  https://ai.google.dev/edge/mediapipe/solutions/vision/pose_landmarker?hl=fr
# Les noms d'origine sont fournis en anglais dans la documentation.
# Ils sont traduits ici en français afin de faciliter l'interprétation des variables et l'analyse des résultats.

# L'index correspond à la position du landmark dans la liste complète des 33 points MediaPipe
# Les landmarks non pertinents sont listés mais commentés afin d'optimiser notre sélection de features

LANDMARK_NAMES = {

    #---------------------------------
    # Landmarks conservés
    #---------------------------------
    # Tête / orientation du regard
    0:  'nez',                    # nose
    2:  'oeil_gauche',            # left_eye
    5:  'oeil_droit',             # right_eye
    7:  'oreille_gauche',         # left_ear
    8:  'oreille_droite',         # right_ear

    # Haut du corps
    11: 'epaule_gauche',          # left_shoulder
    12: 'epaule_droite',          # right_shoulder
    13: 'coude_gauche',           # left_elbow
    14: 'coude_droit',            # right_elbow
    15: 'poignet_gauche',         # left_wrist
    16: 'poignet_droit',          # right_wrist

    # Bas du corps
    23: 'hanche_gauche',          # left_hip
    24: 'hanche_droite',          # right_hip
    25: 'genou_gauche',           # left_knee
    26: 'genou_droit',            # right_knee
    27: 'cheville_gauche',        # left_ankle
    28: 'cheville_droite',        # right_ankle
    29: 'talon_gauche',           # left_heel
    30: 'talon_droit',            # right_heel
    31: 'pied_gauche',            # left_foot_index
    32: 'pied_droit',             # right_foot_index

    #---------------------------------
    # Landmarks ignorés
    #---------------------------------
    # 1:  'oeil_gauche_interieur',  # left_eye_inner
    # 3:  'oeil_gauche_exterieur',  # left_eye_outer
    # 4:  'oeil_droit_interieur',   # right_eye_inner
    # 6:  'oeil_droit_exterieur',   # right_eye_outer
    # 17: 'auriculaire_gauche',     # left_pinky
    # 18: 'auriculaire_droit',      # right_pinky
    # 19: 'index_gauche',           # left_index
    # 20: 'index_droit',            # right_index
    # 21: 'pouce_gauche',           # left_thumb
    # 22: 'pouce_droit'             # right_thumb
    # 9:  'bouche_gauche',          # mouth_left
    # 10: 'bouche_droite',          # mouth_right


}

print(f'✅ {len(LANDMARK_NAMES)} landmarks conservés sur 33')
print(f'   → {len(LANDMARK_NAMES) * 4} features par image (x, y, z, visibilité)')

✅ 21 landmarks conservés sur 33
   → 84 features par image (x, y, z, visibilité)


## Test de détection sur une image

In [4]:
test_path = list((Path(DATASET_DIR) / CLASSES[0]).glob('*.jpg'))[0]
print(f'Image testée : {test_path}')

# Chargement de l'image au format MediaPipe
image_mp = mp.Image.create_from_file(str(test_path))

with vision.PoseLandmarker.create_from_options(options) as landmarker:
    result = landmarker.detect(image_mp)

if result.pose_landmarks and len(result.pose_landmarks) > 0:
    landmarks = result.pose_landmarks[0]
    print('✅ Pose détectée !')
    print(f'   Nombre total de landmarks MediaPipe : {len(landmarks)}')
    print(f'   Landmarks que l\'on conserve        : {len(LANDMARK_NAMES)}')

    # Affichage des coordonnées de nos landmarks conservés
    print('\n   Aperçu des coordonnées :')
    for idx, name in list(LANDMARK_NAMES.items())[:3]:  # affiche les 3 premiers
        lm = landmarks[idx]
        print(f'   {name:20s} → x={lm.x:.3f}, y={lm.y:.3f}, z={lm.z:.3f}, vis={lm.visibility:.3f}')

    # Visualisation — on ne dessine que nos landmarks sélectionnés
    image_bgr   = cv2.imread(str(test_path))
    image_rgb   = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    h, w, _     = image_rgb.shape
    image_annot = image_rgb.copy()

    for idx, name in LANDMARK_NAMES.items():
        lm     = landmarks[idx]
        cx, cy = int(lm.x * w), int(lm.y * h)
        #On annote les points 
        cv2.circle(image_annot, (cx, cy), 6, (0, 255, 0), -1)
        #On ajoute le nom afin de vérifier la cohérence
        cv2.putText(image_annot, name, (cx + 5, cy - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 1)

    plt.figure(figsize=(10, 8))
    plt.imshow(image_annot)
    plt.title('Détection MediaPipe – landmarks sélectionnés')
    plt.axis('off')
    plt.savefig('images/test_mediapipe_selection_feature.png', dpi=100, bbox_inches='tight')
    plt.close()
    print('\n   Visualisation sauvegardée → test_mediapipe.png')
else:
    print('❌ Pose NON détectée sur cette image')

Image testée : dataset\ap_tchagui\aug_0000.jpg
✅ Pose détectée !
   Nombre total de landmarks MediaPipe : 33
   Landmarks que l'on conserve        : 21

   Aperçu des coordonnées :
   nez                  → x=0.638, y=0.139, z=0.234, vis=1.000
   oeil_gauche          → x=0.644, y=0.119, z=0.180, vis=1.000
   oeil_droit           → x=0.651, y=0.118, z=0.221, vis=1.000

   Visualisation sauvegardée → test_mediapipe.png


## Extraction sur tout le dataset

In [5]:
rows          = []
total         = 0
non_detectees = 0

with vision.PoseLandmarker.create_from_options(options) as landmarker:
    for classe in CLASSES:
        dossier      = Path(DATASET_DIR) / classe
        # Récupération de tous les fichiers jpg du dossier
        images_paths = list(dossier.glob('*.jpg'))
        detectees    = 0

        print(f'\n🥋 Traitement : {classe} ({len(images_paths)} images)')

        for img_path in images_paths:
            total += 1
            # Chargement de l'image au format MediaPipe
            image_mp = mp.Image.create_from_file(str(img_path))
            # Détection de la pose
            result   = landmarker.detect(image_mp)
            row      = {}

            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                detectees += 1
                # On récupère le premier (et unique) squelette détecté
                landmarks = result.pose_landmarks[0]

                # Extraction des coordonnées uniquement pour nos landmarks sélectionnés
                # On itère sur le dictionnaire {index: nom} défini en cellule 3
                for idx, name in LANDMARK_NAMES.items():
                    lm = landmarks[idx]
                    row[f'{name}_x']   = lm.x          # position horizontale (0=gauche, 1=droite)
                    row[f'{name}_y']   = lm.y          # position verticale (0=haut, 1=bas)
                    row[f'{name}_z']   = lm.z          # profondeur estimée
                    row[f'{name}_vis'] = lm.visibility # score de confiance (0=invisible, 1=certain)
                row['detection_ok'] = True

            else:
                # Pose non détectée → on remplit avec NaN pour garder la ligne
                non_detectees += 1
                for name in LANDMARK_NAMES.values():
                    for coord in ['x', 'y', 'z', 'vis']:
                        row[f'{name}_{coord}'] = np.nan
                row['detection_ok'] = False

            # Ajout du label et du nom de fichier pour traçabilité
            row['label']   = classe
            row['fichier'] = img_path.name
            rows.append(row)

        print(f'   Détectées : {detectees}/{len(images_paths)}')

# Conversion en DataFrame et sauvegarde
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print(f'\n📊 Résumé extraction')
print(f'   Images traitées   : {total}')
print(f'   Poses détectées   : {total - non_detectees}')
print(f'   Non détectées     : {non_detectees}')
print(f'   Taux de détection : {(total - non_detectees)/total*100:.1f}%')
print(f'\n✅ CSV sauvegardé → {OUTPUT_CSV}')


🥋 Traitement : ap_tchagui (150 images)
   Détectées : 138/150

🥋 Traitement : yop_tchagui (150 images)
   Détectées : 146/150

📊 Résumé extraction
   Images traitées   : 300
   Poses détectées   : 284
   Non détectées     : 16
   Taux de détection : 94.7%

✅ CSV sauvegardé → exports/pose_coordonnes.csv


## Nettoyage et aperçu du CSV

In [6]:
# Chargement du CSV généré
df = pd.read_csv(OUTPUT_CSV)

# Aperçu des colonnes clés — on utilise les noms français définis dans LANDMARK_NAMES
print('📋 Aperçu colonnes clés\n')
cols_apercu = ['fichier', 'label', 'detection_ok',
               'genou_gauche_x', 'genou_gauche_y',
               'genou_droit_x',  'genou_droit_y',
               'cheville_gauche_x', 'cheville_gauche_y']
print(df[cols_apercu].head(10).to_string())

# Distribution des classes pour vérifier l'équilibre du dataset
print(f'\n📊 Distribution des classes :')
print(df['label'].value_counts().to_string())

# Nombre de lignes où MediaPipe n'a pas détecté de pose
print(f'\n⚠️  Lignes sans détection : {df[df["detection_ok"]==False].shape[0]}')

# Suppression des lignes sans détection — inutilisables pour l'entraînement
df_clean = df[df['detection_ok'] == True].copy()

# Suppression des colonnes techniques inutiles pour le modèle
df_clean.drop(columns=['detection_ok', 'fichier'], inplace=True)

# Sauvegarde du dataset propre
df_clean.to_csv('exports/coordonnees_clean.csv', index=False)

print(f'\n✅ Dataset nettoyé sauvegardé')
print(f'   Lignes conservées : {len(df_clean)}')

📋 Aperçu colonnes clés

        fichier       label  detection_ok  genou_gauche_x  genou_gauche_y  genou_droit_x  genou_droit_y  cheville_gauche_x  cheville_gauche_y
0  aug_0000.jpg  ap_tchagui          True        0.421189        0.342786       0.607213       0.683587           0.252499           0.292247
1  aug_0001.jpg  ap_tchagui          True        0.556498        0.465573       0.531499       0.711204           0.651428           0.408756
2  aug_0002.jpg  ap_tchagui         False             NaN             NaN            NaN            NaN                NaN                NaN
3  aug_0003.jpg  ap_tchagui          True        0.358221        0.362019       0.534509       0.705341           0.211572           0.283460
4  aug_0004.jpg  ap_tchagui          True        0.511730        0.813714       0.550380       0.392353           0.503876           0.994938
5  aug_0005.jpg  ap_tchagui          True        0.475954        0.656551       0.441033       0.521892           0.480875  